pivot table

In [34]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('names.csv',header=0,index_col=0,parse_dates=['Dob'],date_format='%Y-%m-%d')
now = pd.Timestamp.now()
df['Age'] = ((now.year - df["Dob"].dt.year))
df = df.assign(Post = np.where(df['Age']<=25,'Junior',np.where(df['Age']<=35,'Senior',np.where(df['Age']>35,'Manager','Intern'))))
df.dtypes
df.to_csv('names.csv',header=True,index=True)

In [ ]:
df = pd.read_csv('names.csv',header=0,index_col=0,parse_dates=['Dob'],dayfirst=False)
df.drop('Post',axis=1,inplace=True)
#df.assign(pd.cut(df['Age'],bins=[0,25,35,float('inf'),labels=['Junior,Senior,Man']]))
df.loc[df['Name']=='Michael','Age'] = 18
df['Post'] = pd.cut(df['Age'],bins=[0,25,35,float('inf')],labels=['Junior','Senior','Manager'])
df['Post'] = df['Post'].cat.add_categories(['Intern']).fillna('Intern') # it is used to add a new category 
df.to_csv('names.csv',header=True,index=True)

In [ ]:
df['Age'].rolling(window=3).mean() # moving average

In [ ]:
df['Post'] = df['Post'].cat.reorder_categories(['Intern','Junior', 'Senior', 'Manager',])
condition = [df['Post']=='Intern',df['Post']=='Junior',df['Post']=='Senior',df['Post']=='Manager']
salary = [20000,30000,40000,50000]
df['Salary'] = np.select(condition,salary)
df.to_csv('names.csv',header=True,index = True)

In [ ]:
df = pd.read_csv('names.csv',index_col = 0 , header=0)
df['cumulative_salary'] = df['Salary'].cumsum()
print(df)
df.pivot_table(index='Post',columns='Department',values='Salary',aggfunc='sum').fillna(0)

In [ ]:
pd.crosstab( df["Department"],df["Post"]) # used to find the frequency of each category

In [ ]:
pd.melt(df, id_vars=["Id"], value_vars=["Salary"]) # id_vars is the list of columns to be used as the index value_vars is the list of columns to be used as the value with thir respective column name to 

In [ ]:
data = {
    'Id' : [1,2,3,4,5],
    'Name' : ['John', 'Mary', 'Alex', 'Peter', 'Anna'],
    'Age' : [20, 25, 30, 35, 40],
    'Salary' : [1200, 2500, 2800, 3600, 4750],
    'Department' : ['HR', 'IT', 'HR', 'IT', 'HR']
}
df = pd.DataFrame(data)
df.set_index('Id',inplace=True)

In [ ]:
df['Rank'] = df.groupby('Department')['Salary'].rank(method='first',ascending=True).astype('Int64')
print(df)
df.drop('Rank', axis=1, inplace=True)

In [ ]:
df['Dept_wise_Sal_Rank'] = df.groupby('Department')['Salary'].rank(method='first', ascending=False).astype(int)
df['Salary_Rank'] = df['Salary'].rank(method='first', ascending=False).astype(int)
df

In [ ]:
df.stack() # stack the columns into row based on the index values
df.unstack() # unstack the columns into rows  based on the column name and the value under the column into rows 

In [ ]:
import pandas as pd
df = pd.DataFrame({
    'Name': ['A', 'B', 'C', 'D', 'E'],
    'Dept': ['IT', 'IT', 'HR', 'HR', 'HR'],
    'Salary': [50, 65, 55, 55, 65]
})
df['Rank'] = df.groupby('Dept')['Salary'].rank(method='first', ascending=False).astype(int)
print(df)
df.sort_values(by=['Dept','Rank'], ascending=[False,True])

  Name Dept  Salary  Rank
0    A   IT      50     2
1    B   IT      65     1
2    C   HR      55     2
3    D   HR      55     3
4    E   HR      65     1


,Name,Dept,Salary,Rank
1,B,IT,65,1
0,A,IT,50,2
4,E,HR,65,1
2,C,HR,55,2
3,D,HR,55,3


In [200]:
df['Dept_Avg'] = df.groupby('Dept')['Salary'].transform('mean').astype('int')
df['Dept'] = df['Dept'].astype('category')

multi hierarchical index

In [185]:
df.set_index(['Dept','Rank'],inplace=True)
df.sort_index(inplace=True)
df.reset_index(inplace=True,drop=False)
#df.drop('index',axis=1,inplace=True)
df

,Dept,Rank,Name,Salary,Dept_Avg
0,HR,1,E,65,58
1,HR,2,C,55,58
2,HR,3,D,55,58
3,IT,1,B,65,57
4,IT,2,A,50,57


advance grouping 

In [186]:
df.groupby('Dept').agg({'Salary':['sum','mean','max']}) # This calculates sum, mean, max salary for each department.
df.groupby('Dept')['Salary'].agg(lambda x : x.max()-x.min()) # This calculates salary spread.
df.groupby('Dept').head(2).sort_values('Salary',ascending=False) # This shows the top 2 departments with highest salary.    
df.query('Salary == 55 and Dept == "HR" ') # This shows all the employees with 55 salary and in HR department.
df['Bonus'] = df.apply(lambda x : x['Salary']*.05 , axis=1) # This applies a function to each row and adds a new column.
df.drop('Bonus',axis=1,inplace=True) # This drops the Bonus column.
df['Bonus'] = df['Salary']*.05
df.drop('Bonus',axis=1,inplace=True)

Method chaining pipelines are a powerful tool in pandas.

You can chain multiple methods together to perform a complex operation.

In [ ]:
new = df.query("Salary > 55").assign(Bonus = lambda x : x['Salary']*1.2).groupby('Dept').agg(Avg_Sal =("Salary","mean"))

,Avg_Sal
Dept,
HR,65.0
IT,65.0


Advance Reshaping

In [201]:
df['Bonus'] = df['Salary']*.05
df.pivot(index='Rank',columns='Dept',values='Name')

Dept,HR,IT
Rank,,
1,E,B
2,C,A
3,D,NaN


custom function using pipe

In [205]:
def apply_bonus(df):
    df['Bonus'] = df['Salary'] * 0.1
    return df
df.pipe(apply_bonus)

,Name,Dept,Salary,Rank,Dept_Avg,Bonus
0,A,IT,50,2,57,5.0
1,B,IT,65,1,57,6.5
2,C,HR,55,2,58,5.5
3,D,HR,55,3,58,5.5
4,E,HR,65,1,58,6.5


You mean COALESCE.

This comes from SQL. In pandas, you use it to take first non-null value.

SQL COALESCE
COALESCE(a, b, c)

Returns first NOT NULL value.

Example:

COALESCE(NULL, NULL, 10, 20) → 10

In [ ]:
data = {'A': [1, 2, 3, 4, np.nan], 'B': [2, np.nan, 6, np.nan, 10]}
df = pd.DataFrame(data)

,A,B
0,1.0,2.0
1,2.0,NaN
2,3.0,6.0
3,4.0,NaN
4,NaN,10.0


In [ ]:
print(df)
print(df.ffill(axis=0)) # here axis=0 means row wise and ffill means forward fill current cell is filled by the cell in the back 
print(df.ffill(axis=1)) # here axis=1 means column wise and ffill means forward fill current cell is filled by the cell in the back
print(df.bfill(axis=0)) # here axis=0 means row wise and bfill means backward fill current cell is filled by the cell in the front
print(df.bfill(axis=1)) # here axis=1 means column wise and bfill means backward fill current cell is filled by the cell in the front

     A     B
0  1.0   2.0
1  2.0   NaN
2  3.0   6.0
3  4.0   NaN
4  NaN  10.0
     A     B
0  1.0   2.0
1  2.0   2.0
2  3.0   6.0
3  4.0   6.0
4  4.0  10.0
     A     B
0  1.0   2.0
1  2.0   2.0
2  3.0   6.0
3  4.0   4.0
4  NaN  10.0
     A     B
0  1.0   2.0
1  2.0   6.0
2  3.0   6.0
3  4.0  10.0
4  NaN  10.0
      A     B
0   1.0   2.0
1   2.0   NaN
2   3.0   6.0
3   4.0   NaN
4  10.0  10.0
